In [68]:
import os

repo_dir = os.environ.get("REPO_DIR")

data_dir = os.path.join(repo_dir, "data/")
code_dir = os.path.join(repo_dir, "code/")


os.chdir(code_dir)

import matplotlib.pyplot as plt
import numpy as np
import scipy.linalg
import pickle
import sklearn 
import sys
import pandas as pd
from importlib import reload

from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge
import seaborn as sns

from scipy.stats import spearmanr

import geopandas as gpd

import warnings

from mosaiks.utils.imports import *

# Key prediction functions are here
from analysis.prediction_utils import (X_matrix_to_demeaned_X,df_to_demeaned_y_vars,
make_train_pred_scatterplot as make_scatterplot, cv_solve, solver_kwargs, get_truth_preds_from_kfold_results,
                             predict_y_from_kfold_dict, generalized_demean)

### Generate ADM2 preds of HDI and components

Methodologically, we match the methods using the NL and IWI downscaling experiments

In [69]:
model_directory = data_dir + "/model_data/"

pop_df = pd.read_pickle(data_dir + "/int/GHS_pop/pop_count_sums_for_ADM2_polygons.p")

path = (model_directory+
           "within_country_rcf_and_nl_demeaned_solve_all_outcomes_country_fold"
           "_DENSE_pop_weight=GHS_VIIRS_hist_bins_GHS_pop_weighted.pkl")

nl_and_rcf_demeaned_kfold_dict = pickle.load(open(path, "rb"))

In [70]:
tasks_to_short_name = {"Sub-national HDI" : "HDI",
        "Life expectancy": "LE", 
         "Mean years schooling":"MYS", 
         "Expected years schooling": "EYS",
         "GNI per capita in thousands of US$ (2011 PPP)": "logGNI",
                      }

tasks = list(tasks_to_short_name.keys())


In [71]:
## Read and demean RCF features

mosaiks_features_direc = data_dir + "/features/mosaiks_features/"
X_adm2 = pd.read_pickle(mosaiks_features_direc + "ADM_2_regions_RCF_global_dense_GHS_POP_pop_weight=True.p").drop(columns = "shapeID")
X_adm1 = pd.read_pickle(mosaiks_features_direc + "ADM_2_regions_RCF_global_dense_aggregated_to_ADM1_GHS_POP_pop_weight=True.p")

X_adm0_not_weighted =X_matrix_to_demeaned_X(X_adm1, return_mean_frame = True)

X_adm2["shapeGroup"] = pd.Series(X_adm2.index).apply(lambda x : x[:3]).to_numpy()
X_adm2_demeaned = generalized_demean(X_adm2, X_adm0_not_weighted, "shapeGroup")

X_adm2.drop(columns = "shapeGroup", inplace=True)

In [72]:
## Read and demean NL features

nl_features_direc = data_dir + "features/nl_features/"

nl_adm1 = pd.read_pickle(nl_features_direc +
                         "GDL_HDI_polygons/viirs_percentile_binned_feats_GHS_pop_weighted_rasterio_method.p")
nl_adm2 = pd.read_pickle(nl_features_direc + 
                         "geoBoundaries_ADM2/viirs_geoBoundaries_ADM2_percentile_binned_feats_GHS_pop_weighted_rasterio_method.p")


nl_adm2["shapeGroup"] = pd.Series(nl_adm2.index).apply(lambda x : x[:3]).to_numpy()

## Make demeaned nl feats at ADM2
### Make demeaned X_adm2

nl_adm0_not_weighted = X_matrix_to_demeaned_X(nl_adm1, return_mean_frame = True)

nl_adm2_demean = generalized_demean(nl_adm2, nl_adm0_not_weighted, "shapeGroup")

nl_adm2_demean = nl_adm2_demean.loc[X_adm2_demeaned.index]


nl_adm2.drop(columns = "shapeGroup", inplace=True)
nl_adm2 = nl_adm2.loc[X_adm2.index]

In [73]:
## Read ADM2 shapfile linked to ADM1 shapefile from GDL
path = data_dir + "/int/ADM2_to_GDL_link/adm2_polygons_linked_to_GDL_adm1.p"
adm2_shp_raw = pd.read_pickle(path)

## Read HDI files used for training
gpdf = pd.read_pickle(data_dir + "/int/GDL_HDI/HDI_ADM1_shapefile_clean.p")

raw = pd.read_pickle( (data_dir + "/int/GDL_HDI/"
                     "HDI_indicators_and_indices_clean.p") )

/share/software/user/open/python/3.9.0/lib/python3.9/pickle.py:1715: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again; shapely 2.1 will not have this compatibility.
  setstate(state)
/share/software/user/open/python/3.9.0/lib/python3.9/pickle.py:1715: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again; shapely 2.1 will not have this compatibility.
  setstate(state)


## Now we generate predictions

In [74]:
for task in tasks:
    print(task)
    predicted_deviations_from_adm0_mean = predict_y_from_kfold_dict(X_adm2_demeaned,
                                                                    nl_and_rcf_demeaned_kfold_dict,
                                                                   task,
                                                                   nl_adm2_demean)


    adm2_shp = adm2_shp_raw.merge(predicted_deviations_from_adm0_mean.rename("predicted_dev_from_adm0"), 
                                                              "left", 
                                                              left_on = "shapeID", right_index=True)

    
    adm2_shp = adm2_shp.merge(raw[task].rename("adm1_mean"),
                              "left", left_on="GDL_ADM1", right_index=True)

    if task == tasks[4]: ## For GNI only, log the task in the raw data prior to centering
         adm2_shp["adm1_mean"] = np.log(adm2_shp["adm1_mean"] )
    
    
    adm1_pred_means = adm2_shp.groupby("GDL_ADM1")["predicted_dev_from_adm0"].mean().rename("mean_of_pred_adm2_obs")
    adm2_shp = adm2_shp.merge(adm1_pred_means, "left", left_on = "GDL_ADM1", right_index=True)
    
    adm2_shp["adj_factor"] = adm2_shp["adm1_mean"] - adm2_shp["mean_of_pred_adm2_obs"]
    
    adm2_shp["adjusted_preds"] = adm2_shp["predicted_dev_from_adm0"] + adm2_shp["adj_factor"]
    
    #Clip to [0,1] for HDI. Can be outside range after re-centering.
    if task == tasks[0]: ## If HDI, we clip
        adm2_shp["adjusted_preds"] = np.clip(adm2_shp["adjusted_preds"],0,1)
    
    adm2_shp.set_index("shapeID", inplace=True)

    
    ## Add population totals
    adm2_shp = adm2_shp.merge(pop_df, how="left", left_index=True,right_index=True)

    ## Drop ireland due to shape data anomalies and inability to verify
    adm2_shp.loc[adm2_shp["shapeGroup"] == "IRL","adjusted_preds"] = np.nan

    if task == tasks[0]: ## If HDI, we save a pickle file to aid other analyses
        #adm2_shp_drop_irl.to_pickle(data_dir + "/preds/hdi_preds_at_adm2.p")

        component_dir_name = ""
    else:
        component_dir_name = "components"
    
    short_name = tasks_to_short_name[task]
    
    adm2_shp[["shapeName","shapeGroup",
          "ADM1_shapeID","GDL_ADM1","percent_overlap_GDL_ADM1",
          "adm1_mean","total_pop","adjusted_preds"]].rename(columns={"adm1_mean":f"adm1_{short_name}_Smits","adjusted_preds":f"predicted_adm2_{short_name}",
                                                                    "total_pop":"est_total_pop"}).to_csv(
        data_dir + f"/preds/{component_dir_name}/{short_name}_preds_at_adm2.csv")

    

Sub-national HDI
Life expectancy
Mean years schooling
Expected years schooling
GNI per capita in thousands of US$ (2011 PPP)
